# 1. Prepare the data

Downloads three Roboflow datasets, translates their labels into one 7-class list, and merges them.

**Why labels need translating:** a YOLO label stores only a class *number*, and each dataset numbers its classes differently. This notebook reads each dataset's own `data.yaml` and maps by class *name*, so nobody has to guess the numbering.

The original downloads are never edited, so you can safely rerun everything.

In [ ]:
!pip install -q roboflow pyyaml pillow matplotlib

## Settings

In [ ]:
import random
import shutil
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import yaml
from PIL import Image, ImageDraw

# Colab's working folder is /content; anywhere else, use the current folder.
ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
RAW_DIR = ROOT / 'datasets' / 'raw'        # untouched downloads
MERGED_DIR = ROOT / 'datasets' / 'merged'  # rebuilt from scratch on every run

# The order here defines the class numbers your model will use.
CLASSES = ['coal ore', 'diamond ore', 'gold ore', 'lapis ore', 'redstone ore', 'iron ore', 'emerald ore']

# `splits` says where each source split goes in the merged dataset.
# TODO: `360` has train images only. If these are your own images, the brief
# suggests using them as test data: change this to {'train': 'test'}.
DATASETS = [
    {'name': 'oblig10', 'workspace': 'oblig10', 'project': 'minecraft-ore-detection-20pzg-xejw6', 'version': 2,
     'splits': {'train': 'train', 'valid': 'valid', 'test': 'test'}},
    {'name': 'blocks', 'workspace': 'main-data', 'project': 'minecraft-blocks', 'version': 1,
     'splits': {'train': 'train', 'valid': 'valid', 'test': 'test'}},
    {'name': '360', 'workspace': 'aasmaa-alibhai', 'project': '360-3wqmv', 'version': 1,
     'splits': {'train': 'train'}},
]

## Download

Add your **own** Roboflow API key first:

- **Colab:** key icon in the left sidebar, add a secret named `ROBOFLOW_API_KEY`, and allow notebook access.
- **Elsewhere:** set a `ROBOFLOW_API_KEY` environment variable.

Never paste the key into a cell, because it would be committed with the notebook.

In [ ]:
import os

from roboflow import Roboflow

try:
    from google.colab import userdata
    api_key = userdata.get('ROBOFLOW_API_KEY')
except ImportError:
    api_key = os.environ['ROBOFLOW_API_KEY']

rf = Roboflow(api_key=api_key)
for d in DATASETS:
    project = rf.workspace(d['workspace']).project(d['project'])
    project.version(d['version']).download('yolov8', location=str(RAW_DIR / d['name']))

## Translate class names

Check this printout before moving on. Each original name should sit next to the ore you expect.

In [ ]:
def to_master_id(name):
    """Return the CLASSES number for a dataset's class name, e.g. 'Deepslate RedstoneOre' -> 4."""
    matches = [i for i, cls in enumerate(CLASSES) if cls.split()[0] in name.lower()]
    if len(matches) != 1:
        raise ValueError(f'Cannot match class name {name!r} to exactly one of {CLASSES}')
    return matches[0]


def build_remap(dataset_dir):
    names = yaml.safe_load((dataset_dir / 'data.yaml').read_text())['names']
    if isinstance(names, dict):  # YOLO allows {0: 'coal'} as well as ['coal']
        names = [names[i] for i in sorted(names)]
    return {i: to_master_id(name) for i, name in enumerate(names)}, names


for d in DATASETS:
    remap, names = build_remap(RAW_DIR / d['name'])
    print(f"\n{d['name']}")
    for old_id, name in enumerate(names):
        print(f'  {old_id}: {name:<25} -> {remap[old_id]}: {CLASSES[remap[old_id]]}')

## Merge

Filenames get a dataset prefix so images from different datasets can't overwrite each other. An image with no label file is kept as a background image, which tells YOLO there's nothing to detect in it.

In [ ]:
def merge_dataset(d):
    source = RAW_DIR / d['name']
    remap, _ = build_remap(source)
    for source_split, target_split in d['splits'].items():
        images_dir = source / source_split / 'images'
        labels_dir = source / source_split / 'labels'
        if not images_dir.exists():
            print(f"  {d['name']}/{source_split}: no images, skipped")
            continue

        out_images = MERGED_DIR / target_split / 'images'
        out_labels = MERGED_DIR / target_split / 'labels'
        out_images.mkdir(parents=True, exist_ok=True)
        out_labels.mkdir(parents=True, exist_ok=True)

        count = 0
        for image_path in sorted(images_dir.iterdir()):
            new_name = f"{d['name']}_{image_path.name}"
            shutil.copy(image_path, out_images / new_name)
            count += 1

            label_path = labels_dir / f'{image_path.stem}.txt'
            if not label_path.exists():
                continue
            new_lines = []
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if not parts:
                    continue
                if len(parts) != 5:
                    raise ValueError(f'Expected 5 values in {label_path}: {line!r}')
                parts[0] = str(remap[int(parts[0])])
                new_lines.append(' '.join(parts))
            (out_labels / f'{Path(new_name).stem}.txt').write_text('\n'.join(new_lines) + '\n' if new_lines else '')
        print(f"  {d['name']}/{source_split} -> {target_split}: {count} images")


if MERGED_DIR.exists():
    shutil.rmtree(MERGED_DIR)
for d in DATASETS:
    merge_dataset(d)

data_yaml = {'path': str(MERGED_DIR), 'train': 'train/images', 'val': 'valid/images', 'names': dict(enumerate(CLASSES))}
if (MERGED_DIR / 'test' / 'images').exists():
    data_yaml['test'] = 'test/images'
(MERGED_DIR / 'data.yaml').write_text(yaml.safe_dump(data_yaml, sort_keys=False))
print(f"\nWrote {MERGED_DIR / 'data.yaml'}")

## Check the class balance

Very low counts for a class usually mean weak results for that class.

In [ ]:
def count_boxes(split):
    counts = Counter()
    for label_path in (MERGED_DIR / split / 'labels').glob('*.txt'):
        for line in label_path.read_text().splitlines():
            counts[int(line.split()[0])] += 1
    return counts


splits = [s for s in ('train', 'valid', 'test') if (MERGED_DIR / s).exists()]
all_counts = {s: count_boxes(s) for s in splits}
print(f"{'class':<15}" + ''.join(f'{s:>8}' for s in splits))
for i, cls in enumerate(CLASSES):
    print(f'{cls:<15}' + ''.join(f'{all_counts[s][i]:>8}' for s in splits))

## Spot-check the labels

The most important check in this notebook: **does each box name match the ore inside it?** Rerun this cell a few times. Any wrong names mean a bad translation, so fix that before training.

This adapts `draw_yolo_bbox` from the Week 6 tutorial.

In [ ]:
def draw_labels(image_path, label_path):
    image = Image.open(image_path).convert('RGB')
    width, height = image.size
    draw = ImageDraw.Draw(image)
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            class_id, x, y, w, h = line.split()
            x, y, w, h = float(x) * width, float(y) * height, float(w) * width, float(h) * height
            box = (x - w / 2, y - h / 2, x + w / 2, y + h / 2)
            draw.rectangle(box, outline='lime', width=3)
            draw.text((box[0] + 3, box[1] + 3), CLASSES[int(class_id)], fill='lime')
    return image


def show_samples(split='train', per_dataset=3):
    for d in DATASETS:
        images = sorted((MERGED_DIR / split / 'images').glob(f"{d['name']}_*"))
        if not images:
            continue
        samples = random.sample(images, min(per_dataset, len(images)))
        fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5), squeeze=False)
        for ax, image_path in zip(axes[0], samples):
            label_path = MERGED_DIR / split / 'labels' / f'{image_path.stem}.txt'
            ax.imshow(draw_labels(image_path, label_path))
            ax.axis('off')
        fig.suptitle(d['name'])
        plt.show()


show_samples()

## Save your work

Each Colab notebook gets its own temporary machine, and `/content` is wiped when it shuts down. This saves the merged dataset to Google Drive, where `train.ipynb` picks it up. Outside Colab, it does nothing.

In [ ]:
try:
    from google.colab import drive
except ImportError:
    print('Not in Colab: train.ipynb will read datasets/merged directly.')
else:
    drive.mount('/content/drive')
    drive_dir = Path('/content/drive/MyDrive/minecraft-ore-yolo')
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.make_archive(str(drive_dir / 'merged'), 'zip', MERGED_DIR)
    print(f"Saved {drive_dir / 'merged.zip'}")